In [116]:
import h5py
import numpy as np

input_file = './outputs/TOI-776_b_assignment3_taskB_spectrum.h5'
output_file = './outputs/TOI-776_b_observed_spectrum.dat'

FIXED_ERROR = 1e-5

with h5py.File(input_file, 'r') as f:
    wl = f['Output/Spectra/native_wlgrid'][:]
    spectrum = f['Output/Spectra/native_spectrum'][:]

    # Fixed absolute uncertainty
    noise = np.full_like(spectrum, FIXED_ERROR)

    np.random.seed(42)
    observed_spectrum = spectrum + np.random.normal(0, noise)

    data = np.column_stack([wl, observed_spectrum, noise])

    np.savetxt(
        output_file,
        data,
        fmt='%.8e',
        header='wavelength(um) transit_depth error'
    )

print(f"✓ Created: {output_file}")
print(f"Wavelength range: {wl.min():.3f} - {wl.max():.3f} microns")
print(f"Number of points: {len(wl)}")
print(f"Fixed error used: {FIXED_ERROR}")


✓ Created: ./outputs/TOI-776_b_observed_spectrum.dat
Wavelength range: 0.300 - 50.002 microns
Number of points: 76744
Fixed error used: 1e-05


In [147]:
import subprocess
import os
import glob

os.chdir('/ca25/comp_astro_25/Assigment_3/TOI-776_b')

print("Running retrieval...")

# NO --output-size flag here! It should be in the .par file
subprocess.run([
    "taurex", 
    "-i", "TOI-776_b_transmission_2.par", 
    "--retrieval"
])



Running retrieval...


taurex - INFO - TauREx 3.3.0
taurex - INFO - TauREx PROGRAM START AT 2025-12-23 20:37:02.173827
taurex.ParamParser - INFO - Interpolation mode set to linear
taurex.ParamParser - WARNING - Xsecs will be loaded in memory
taurex.ParamParser - WARNING - Radis is disabled
taurex.ParamParser - WARNING - Radis default grid will be used
taurex.ClassFactory - INFO - Reloading all modules and plugins
taurex.ClassFactory - INFO - ----------Plugin loading---------
taurex.TransmissionModel - INFO - Building model........
taurex.TransmissionModel - INFO - Collecting paramters
taurex.TransmissionModel - INFO - Setting up profiles
taurex.TransmissionModel - INFO - Setting up contributions
taurex.TransmissionModel - INFO - DONE
taurex.TransmissionModel - INFO - Computing pressure profile
taurex.ChemistryModel - INFO - Initializing chemistry model
taurex.OpacityCache - INFO - Reading opacity H2O
taurex.OpacityCache - INFO - Loading opacity H2O into model
taurex.OpacityCache - INFO - Reading opacity CH4


it=   464 logz=763409.8417385148

/root/anaconda3/lib/python3.13/site-packages/nestle.py:206: RuntimeWarning: divide by zero encountered in scalar divide
  cov = wsum / (wsum**2 - w2sum) * np.einsum('i,ij,ik', weights, dx, dx)
/root/anaconda3/lib/python3.13/site-packages/nestle.py:206: RuntimeWarning: invalid value encountered in multiply
  cov = wsum / (wsum**2 - w2sum) * np.einsum('i,ij,ik', weights, dx, dx)
taurex.Nestle - INFO - Sampling time 1150.627879858017 s
taurex.Nestle - INFO - Post-processing - Generating spectra and profiles
taurex.Nestle - INFO - Computing solution 0
taurex.TransmissionModel - INFO - Computing pressure profile
taurex.ChemistryModel - INFO - Initializing chemistry model
taurex.Absorption - INFO - Using cross-sections? True
taurex.Absorption - INFO - Recomputing active gas H2O opacity
taurex.Absorption - INFO - Recomputing active gas CH4 opacity
taurex.Absorption - INFO - Recomputing active gas CO2 opacity
taurex.Absorption - INFO - Recomputing active gas CO opacity
taurex.Absorption - INFO

niter: 465
ncall: 914
nsamples: 545
logz: 773491.718 +/-  0.357
h: 10.195


Traceback (most recent call last):
  File "/root/anaconda3/bin/taurex", line 8, in <module>
    sys.exit(main())
             ~~~~^^
  File "/root/anaconda3/lib/python3.13/site-packages/taurex/taurex.py", line 636, in main
    solution = optimizer.fit(output_size=output_size)
  File "/root/anaconda3/lib/python3.13/site-packages/taurex/optimizer/optimizer.py", line 539, in fit
    solution = self.generate_solution(output_size=output_size)
  File "/root/anaconda3/lib/python3.13/site-packages/taurex/optimizer/optimizer.py", line 702, in generate_solution
    profile_dict, spectrum_dict = self.generate_profiles(
                                  ~~~~~~~~~~~~~~~~~~~~~~^
        solution, self._observed.wavenumberGrid
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/root/anaconda3/lib/python3.13/site-packages/taurex/optimizer/optimizer.py", line 653, in generate_profiles
    return self._model.compute_error(
           ~~~~~~~~~~~~~~~~~~~~~~~~~^
        sample_iter, wngri


Checking for output files:
  ✗ No output files found

All files in outputs:
    ./outputs/TOI-776_b_assignment3_taskB_spectrum.dat
    ./outputs/TOI-776_b_observed_spectrum.dat
    ./outputs/TOI-776_b_assignment3_taskB_spectrum.png
    ./outputs/TOI-776_b_assignment3_taskB_spectrum.h5


In [ ]:

import corner

chain = pd.read_csv("outputs/chain.txt", delim_whitespace=True)

# Inspect columns
print(chain.columns)

# Select only retrieved parameters
retrieved_params = ["planet_radius", "T", "H2O"]
samples = chain[retrieved_params].values

# Corner plot
fig = corner.corner(
    samples,
    labels=retrieved_params,
    show_titles=True,
    quantiles=[0.16, 0.5, 0.84],
    title_fmt=".3f"
)
plt.savefig("outputs/posterior_plot.png", dpi=150)
plt.show()


In [ ]:
# Observed spectrum
obs = np.loadtxt("outputs/TOI-776_b_observed_spectrum.dat")
wavelength, flux_obs, flux_err = obs.T

# Retrieved spectrum
model = np.loadtxt("outputs/TOI-776_b_taskD_bestfit.dat")
wavelength_model, flux_model = model.T

plt.errorbar(wavelength, flux_obs, yerr=flux_err, fmt="o", label="Observed")
plt.plot(wavelength_model, flux_model, label="Retrieved Best-fit", color="red")
plt.xlabel("Wavelength [micron]")
plt.ylabel("Transit Depth")
plt.legend()
plt.savefig("outputs/spectrum_comparison.png", dpi=150)
plt.show()
